# Tuần 06: Thống kê nhập môn cho Results

Mục tiêu: dùng cleaned pre/post data để tính `n`, `mean`, `SD`, `SE`, `95% CI`, rồi viết một Results paragraph thận trọng. Tuần này ưu tiên diễn giải kết quả, không biến p-value thành mục tiêu chính.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib

import pandas as pd
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["svg.hashsalt"] = "week06-intro-statistics"
import matplotlib.pyplot as plt
from scipy import stats

THIS_WEEK = "week-06-intro-statistics-for-results"
EXPECTED_SHA256 = "175469cd9120b36a467d0e0b439f78859525841555c6a30bc5de0777edb9137a"


def find_week_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "weeks" / THIS_WEEK,
        Path.cwd().parent,
        Path.cwd().parent / "weeks" / THIS_WEEK,
    ]
    for candidate in candidates:
        if candidate.name == THIS_WEEK and (candidate / "data/raw").exists():
            return candidate
        if (candidate / "data/raw/week06_tcsol_prepost_scores.csv").exists():
            return candidate
    week_dir = Path.cwd() / "weeks" / THIS_WEEK
    week_dir.mkdir(parents=True, exist_ok=True)
    return week_dir


def course_path(path):
    path = Path(path)
    if THIS_WEEK in path.parts:
        start = path.parts.index(THIS_WEEK)
        return Path("weeks") / Path(*path.parts[start:])
    return path.name


WEEK_DIR = find_week_dir()
DATA_PATH = WEEK_DIR / "data/raw/week06_tcsol_prepost_scores.csv"
TABLE_DIR = WEEK_DIR / "outputs/tables"
FIGURE_DIR = WEEK_DIR / "outputs/figures"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    source_url = "https://raw.githubusercontent.com/mtuann/tcsol-python-research/main/weeks/week-06-intro-statistics-for-results/data/raw/week06_tcsol_prepost_scores.csv"
    urlretrieve(source_url, DATA_PATH)

actual_hash = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
if actual_hash != EXPECTED_SHA256:
    raise ValueError("Downloaded CSV does not match the expected Week 06 teaching dataset.")

print("pandas version:", pd.__version__)
print("matplotlib version:", matplotlib.__version__)
print("Data file:", course_path(DATA_PATH))


pandas version: 2.3.3
matplotlib version: 3.9.4
Data file: weeks/week-06-intro-statistics-for-results/data/raw/week06_tcsol_prepost_scores.csv


## 1. Khung nghiên cứu

Research question nhỏ: **Sau một hoạt động Hán ngữ ngắn hạn, average gain là bao nhiêu và estimate này chắc đến mức nào?**

Paper connection: Week 05 tạo figure. Week 06 thêm uncertainty để câu Results không chỉ nói “cao hơn/thấp hơn”, mà nói rõ `N`, khoảng ước lượng và limitation.

In [2]:
df = pd.read_csv(DATA_PATH)
usable = df[df["usable_pre_post"] == True].copy()
usable["activity_label"] = usable["activity_focus"].map({
    "result_complements": "Result complements",
    "measure_words": "Measure words",
    "vocabulary_review": "Vocabulary review",
    "word_order": "Word order",
})

print("Raw rows:", len(df))
print("Usable rows:", len(usable))
print(usable[["learner_id", "activity_focus", "pre_score", "post_score", "gain_score"]].head(8).to_string(index=False))


Raw rows: 36
Usable rows: 25
learner_id     activity_focus  pre_score  post_score  gain_score
      S001      measure_words       62.0        75.0        13.0
      S002      measure_words       58.0        70.0        12.0
      S005 result_complements       55.0        68.0        13.0
      S006 result_complements       59.0        73.0        14.0
      S007 result_complements       61.0        74.0        13.0
      S009         word_order       64.0        73.0         9.0
      S010         word_order       63.0        72.0         9.0
      S011         word_order       65.0        74.0         9.0


## 2. Bốn con số cần đọc

- `mean`: mức gain trung bình trong sample.
- `SD`: learner trong nhóm khác nhau nhiều hay ít.
- `SE`: estimate mean ổn định đến đâu.
- `95% CI`: một khoảng ước lượng hợp lý cho mean, nếu giả định thống kê phù hợp.

Mental model: **SD nói về người học; SE/CI nói về estimate mean.**

In [3]:
order = ["result_complements", "measure_words", "vocabulary_review", "word_order"]
rows = []

for activity in order:
    group = usable[usable["activity_focus"] == activity]
    n = len(group)
    mean_gain = group["gain_score"].mean()
    sd_gain = group["gain_score"].std(ddof=1)
    se_gain = group["gain_score"].sem()
    t_critical = stats.t.ppf(0.975, n - 1)
    margin = t_critical * se_gain
    rows.append({
        "activity_focus": activity,
        "activity_label": group["activity_label"].iloc[0],
        "n": n,
        "pre_mean": group["pre_score"].mean(),
        "post_mean": group["post_score"].mean(),
        "mean_gain": mean_gain,
        "sd_gain": sd_gain,
        "se_gain": se_gain,
        "t_critical_95": t_critical,
        "ci95_low": mean_gain - margin,
        "ci95_high": mean_gain + margin,
    })

activity_stats = pd.DataFrame(rows)
rounded = activity_stats.copy()
number_cols = ["pre_mean", "post_mean", "mean_gain", "sd_gain", "se_gain", "t_critical_95", "ci95_low", "ci95_high"]
rounded[number_cols] = rounded[number_cols].round(2)
output_path = TABLE_DIR / "week06_activity_statistics.csv"
rounded.to_csv(output_path, index=False)

print(rounded[["activity_label", "n", "mean_gain", "sd_gain", "se_gain", "ci95_low", "ci95_high"]].to_string(index=False))
print("Saved:", course_path(output_path))


    activity_label  n  mean_gain  sd_gain  se_gain  ci95_low  ci95_high
Result complements  6      13.00     0.89     0.37     12.06      13.94
     Measure words  5      12.60     0.55     0.24     11.92      13.28
 Vocabulary review  8       9.75     0.71     0.25      9.16      10.34
        Word order  6       8.83     0.41     0.17      8.40       9.26
Saved: weeks/week-06-intro-statistics-for-results/outputs/tables/week06_activity_statistics.csv


## 3. Đọc bảng như người viết Results

Đừng bắt đầu bằng “significant hay không?”. Hãy đọc theo thứ tự:

1. `n`: có bao nhiêu record usable?
2. `mean_gain`: estimate chính là gì?
3. `ci95_low` đến `ci95_high`: khoảng ước lượng rộng hay hẹp?
4. limitation: sample nhỏ, synthetic data, không random assignment.

In [4]:
n = len(usable)
mean_gain = usable["gain_score"].mean()
sd_gain = usable["gain_score"].std(ddof=1)
se_gain = usable["gain_score"].sem()
t_critical = stats.t.ppf(0.975, n - 1)
ci_low = mean_gain - t_critical * se_gain
ci_high = mean_gain + t_critical * se_gain

overall_summary = pd.DataFrame([{
    "raw_n": len(df),
    "usable_n": n,
    "pre_mean": usable["pre_score"].mean(),
    "post_mean": usable["post_score"].mean(),
    "mean_gain": mean_gain,
    "sd_gain": sd_gain,
    "se_gain": se_gain,
    "t_critical_95": t_critical,
    "ci95_low": ci_low,
    "ci95_high": ci_high,
    "min_gain": usable["gain_score"].min(),
    "max_gain": usable["gain_score"].max(),
}])
overall_rounded = overall_summary.round(2)
overall_path = TABLE_DIR / "week06_overall_gain_summary.csv"
overall_rounded.to_csv(overall_path, index=False)

print(overall_rounded.to_string(index=False))
print("Saved:", course_path(overall_path))


 raw_n  usable_n  pre_mean  post_mean  mean_gain  sd_gain  se_gain  t_critical_95  ci95_low  ci95_high  min_gain  max_gain
    36        25     64.68      75.56      10.88      1.9     0.38           2.06      10.1      11.66       8.0      14.0
Saved: weeks/week-06-intro-statistics-for-results/outputs/tables/week06_overall_gain_summary.csv


## 4. Figure có confidence interval

Figure Week 06 không cần màu mè hơn Week 05. Nó chỉ thêm một thứ: thanh interval để người đọc thấy estimate mean có độ bất định.

In [5]:
png_path = FIGURE_DIR / "week06_mean_gain_ci_by_activity.png"
svg_path = FIGURE_DIR / "week06_mean_gain_ci_by_activity.svg"
palette = ["#1f7a4d", "#2563eb", "#b8325f", "#b45309"]
figure_metadata = {"Date": "2026-06-03"}

fig, ax = plt.subplots(figsize=(8.6, 5.2))
y_positions = range(len(activity_stats))
means = activity_stats["mean_gain"]
error_left = means - activity_stats["ci95_low"]
error_right = activity_stats["ci95_high"] - means

ax.barh(y_positions, means, color=palette, alpha=0.88, height=0.55)
ax.errorbar(means, y_positions, xerr=[error_left, error_right], fmt="none", ecolor="#172033", elinewidth=1.8, capsize=5)
ax.set_yticks(list(y_positions))
ax.set_yticklabels([f"{label}\n(n={n})" for label, n in zip(activity_stats["activity_label"], activity_stats["n"])])
ax.invert_yaxis()
ax.set_xlabel("Mean gain score with 95% CI")
ax.set_title("Estimated mean gain by activity focus", weight="bold", pad=14)
ax.set_xlim(0, 15)
ax.grid(axis="x", color="#dbe4f0", linewidth=0.8)
ax.grid(axis="y", visible=False)
for idx, row in activity_stats.iterrows():
    ax.text(row["ci95_high"] + 0.2, idx, f"{row['mean_gain']:.2f}", va="center", ha="left", fontsize=10, weight="bold")
fig.text(0.12, 0.035, "Synthetic teaching dataset; intervals describe uncertainty around group mean gains, not causal effects.", color="#5b6578", fontsize=9.5)
fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.savefig(png_path, dpi=300, bbox_inches="tight", metadata=figure_metadata)
fig.savefig(svg_path, bbox_inches="tight", metadata=figure_metadata)
plt.close(fig)

print("Saved PNG:", course_path(png_path))
print("Saved SVG:", course_path(svg_path))


Saved PNG: weeks/week-06-intro-statistics-for-results/outputs/figures/week06_mean_gain_ci_by_activity.png
Saved SVG: weeks/week-06-intro-statistics-for-results/outputs/figures/week06_mean_gain_ci_by_activity.svg


## 5. Caption và Results paragraph frame

Figure caption frame:

> Figure 1. Mean gain score by activity focus in the synthetic Week 06 TCSOL dataset (`N = 25` usable learner records). Error bars show 95% confidence intervals around group mean gains; the figure is descriptive and does not establish causal effects.

Mẫu Results 120-160 từ:

> In the usable Week 06 records (`N = 25`), learners gained an average of `[mean]` points from pre-test to post-test (`SD = [SD]`, `95% CI [low, high]`). By activity focus, `[highest group]` had the highest descriptive mean gain, while `[lowest group]` had the lowest. These estimates should be read cautiously because the dataset is synthetic, group sizes are small, and activity focus was not randomly assigned.

In [6]:
high = activity_stats.sort_values("mean_gain", ascending=False).iloc[0]
low = activity_stats.sort_values("mean_gain", ascending=True).iloc[0]
result_paragraph = (
    f"In the usable Week 06 records (N = {n}), learners gained an average of "
    f"{mean_gain:.2f} points from pre-test to post-test (SD = {sd_gain:.2f}, "
    f"95% CI [{ci_low:.2f}, {ci_high:.2f}]). By activity focus, "
    f"{high['activity_label']} had the highest descriptive mean gain "
    f"(M = {high['mean_gain']:.2f}), while {low['activity_label']} had the lowest "
    f"(M = {low['mean_gain']:.2f}). These estimates should be read cautiously because "
    "the dataset is synthetic, group sizes are small, and activity focus was not randomly assigned. "
    "The result therefore supports a descriptive claim about observed learner records, not a causal claim."
)
print(result_paragraph)


In the usable Week 06 records (N = 25), learners gained an average of 10.88 points from pre-test to post-test (SD = 1.90, 95% CI [10.10, 11.66]). By activity focus, Result complements had the highest descriptive mean gain (M = 13.00), while Word order had the lowest (M = 8.83). These estimates should be read cautiously because the dataset is synthetic, group sizes are small, and activity focus was not randomly assigned. The result therefore supports a descriptive claim about observed learner records, not a causal claim.


## 6. Optional: paired t-test

T-test trả lời câu hỏi khác với CI. CI giúp viết **estimate + uncertainty**. T-test hỏi liệu mean difference có xa 0 không dưới giả định thống kê. Với người mới, chỉ đọc output này như stretch, không dùng p-value một mình để kết luận.

In [7]:
paired_test = stats.ttest_rel(usable["post_score"], usable["pre_score"])
print("paired t statistic:", round(float(paired_test.statistic), 2))
print("p-value:", f"{float(paired_test.pvalue):.3g}")
print("df:", int(paired_test.df))


paired t statistic: 28.63
p-value: 4.58e-20
df: 24


## 7. Transfer sang các hướng nghiên cứu

- TCSOL: report mean gain + CI cho pre/post score.
- Đối chiếu Hán-Việt: report mean difficulty rating + CI theo hiện tượng ngữ pháp.
- MT/MTPE: report mean edit time hoặc error count + CI theo system.
- Chính sách giáo dục: report mean/median coding score theo giai đoạn, nhớ nói rõ đơn vị phân tích là văn bản hay chính sách.

## 8. Exercise

1. Change one activity label to Vietnamese.
2. Rerun the activity statistics table.
3. Save the CI figure.
4. Write a 120-160 word Results paragraph.
5. Add one limitation sentence that does **not** sound like an apology.